# DTFA on Kaggle (VOC-Fog), background training

**One-time setup**
1. Kaggle → Datasets → New Dataset: upload `DTFA_colab.zip`, a zip of the current repo (see README.md, section 3.2). Name it e.g. `dtfa-data`. Only if your uploaded zip predates Dynamic Bridging Tuning, also add `DTFA_code_update.zip` (the updated code files) via *New Version*.
2. New Notebook → Add Input → your `dtfa-data` dataset.
3. Settings: **Accelerator = GPU T4 x2** (the script uses one GPU; P100 can fail with newer PyTorch), Persistence = Files only, Internet can stay off.
4. Edit the parameters in the next cell, then **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers, so you can close the laptop.

**Chunks (12 h limit per run):** run 1 with `STOP_EPOCH = 75`. When it finishes, make run 2: *Add Input → Notebook Output of this notebook (the previous version)*, set `STOP_EPOCH = 100`. The notebook finds the newest checkpoint in the inputs and resumes from it. Keep `EPOCHS`, `TRAIN_SUBSET` and `VAL_SUBSET` the same in every run.

Checkpoints are saved to `/kaggle/working/logs` (every 5 epochs) and appear under the version's Output tab.

**Baseline vs Dynamic Bridging Tuning (DBT):** `DBT = 0` trains the original DTFA, `DBT = 1` the DBT extension (see `DBT.md`). For a fair comparison run both with identical `EPOCHS`, `TRAIN_SUBSET`, `VAL_SUBSET` and `BATCH_SIZE`, in separate notebooks (or versions that do not take each other's output as input), and compare the mAP files. A DBT run also writes `logs/dbt_stats.csv` (per-epoch gate statistics), `logs/dbt_epNNN.pth` (gate checkpoints, needed to resume) and, after the last epoch, `dbt_vis/*.png` (severity / IRT-trust maps).

In [ ]:
# ---- parameters: edit these ----
EPOCHS       = 100     # total epochs; the lr schedule spans this many. Keep the same across chunks.
STOP_EPOCH   = 75      # stop this run after this epoch (multiple of SAVE_PERIOD). Use EPOCHS for the last chunk.
TRAIN_SUBSET = 3000    # number of training images (0 = all 8,513)
VAL_SUBSET   = 600     # number of validation images (0 = all 2,129)
BATCH_SIZE   = 8       # use 4 on CUDA out-of-memory
SAVE_PERIOD  = 5
NUM_WORKERS  = 4

# ---- Dynamic Bridging Tuning ----
DBT          = 1           # 0 = original DTFA (baseline), 1 = DBT
DBT_MODE     = 'spatial'   # 'spatial' (per-location alpha), 'image' (one alpha per image), 'fixed' (alpha = 0.5)
DBT_USE_S, DBT_USE_W, DBT_USE_D = 1, 1, 1   # gate inputs: student feature, severity map W, teacher disagreement D

## 1. Check the GPU works (fails fast if it doesn't)

In [ ]:
import torch, subprocess
print(torch.__version__, torch.version.cuda)
assert torch.cuda.is_available(), 'No GPU: turn on Accelerator in the notebook settings'
print(torch.cuda.get_device_name(0))
x = torch.randn(256, 256, device='cuda'); print('CUDA matmul ok:', (x @ x).sum().item() != 0)

## 2. Unpack the repo into temporary disk (not saved as output)

In [ ]:
import os, glob, shutil, subprocess
tmp = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'
work = os.path.join(tmp, 'work'); os.makedirs(work, exist_ok=True)

zips = glob.glob('/kaggle/input/**/DTFA_colab.zip', recursive=True)
if zips:
    print('unzipping', zips[0])
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', work], check=True)
    repo = os.path.join(work, 'DTFA')
else:  # Kaggle may have auto-extracted the zip when the dataset was created
    marks = glob.glob('/kaggle/input/**/train/train_AWD.py', recursive=True)
    assert marks, 'DTFA_colab.zip or the extracted DTFA folder not found under /kaggle/input'
    src = os.path.dirname(os.path.dirname(marks[0]))
    repo = os.path.join(work, 'DTFA')
    shutil.copytree(src, repo, dirs_exist_ok=True)
print('repo:', repo)

# overlay the updated code (DTFA_code_update.zip: train_AWD.py, utils/fit.py, utils/DBT/, tools/)
upd = glob.glob('/kaggle/input/**/DTFA_code_update.zip', recursive=True)
if upd:
    subprocess.run(['unzip', '-q', '-o', upd[0], '-d', work], check=True); print('code update:', upd[0])
else:  # Kaggle may have auto-extracted it
    marks = glob.glob('/kaggle/input/**/utils/DBT/loss.py', recursive=True)
    if marks:
        src = os.path.dirname(os.path.dirname(os.path.dirname(marks[0])))
        shutil.copytree(src, repo, dirs_exist_ok=True); print('code update:', src)
assert os.path.exists(os.path.join(repo, 'utils', 'DBT', 'loss.py')), 'add DTFA_code_update.zip to the dataset'

# quick data sanity check (an older DTFA_colab.zip had 4 broken images)
from PIL import Image
empty = [p for p in glob.glob(os.path.join(repo, 'datasets/voc_*/*/JPEGImages/*.jpg')) if os.path.getsize(p) == 0]
for i in ['voc_clean/train/JPEGImages/2007_009331', 'voc_clean/train/JPEGImages/2008_000217',
          'voc_clean/train/JPEGImages/2008_000027', 'voc_fog/train/JPEGImages/2007_009327']:
    try:
        Image.open(os.path.join(repo, 'datasets', i + '.jpg')).load()
    except Exception:
        empty.append(i)
assert not empty, f'broken images in the uploaded zip: {empty[:5]} -> upload the current DTFA_colab.zip'

os.chdir(repo)
os.makedirs('/kaggle/working/logs', exist_ok=True)
if os.path.isdir('logs') and not os.path.islink('logs'): shutil.rmtree('logs')
if not os.path.lexists('logs'): os.symlink('/kaggle/working/logs', 'logs')
os.environ['PYTHONPATH'] = repo
print(os.listdir('model_data'))

## 3. Resume point (picks the newest checkpoint from a previous run, if added as input)

In [ ]:
import re
cands = [p for p in glob.glob('/kaggle/input/**/ep*.pth', recursive=True) + glob.glob('/kaggle/working/logs/ep*.pth')]
def ep_of(p): return int(re.search(r'ep(\d+)-', os.path.basename(p)).group(1))
if cands:
    last = max(cands, key=ep_of)
    INIT_EPOCH, MODEL_PATH = ep_of(last), last
    print('resuming from', last, '-> INIT_EPOCH =', INIT_EPOCH)
else:
    INIT_EPOCH, MODEL_PATH = 0, 'model_data/yolox_s.pth'
    print('fresh start from yolox_s.pth')
assert INIT_EPOCH < STOP_EPOCH, 'Nothing to do: checkpoint epoch >= STOP_EPOCH'
DBT_PATH = ''
if DBT and INIT_EPOCH > 0:  # gate checkpoint saved with that epoch
    g = [p for p in glob.glob('/kaggle/input/**/dbt_ep%03d.pth' % INIT_EPOCH, recursive=True)
         + glob.glob('/kaggle/working/logs/dbt_ep%03d.pth' % INIT_EPOCH)]
    DBT_PATH = g[0] if g else ''
    print('DBT gate checkpoint:', DBT_PATH or 'NOT FOUND (gate restarts)')

## 4. Train
Check the log for `matched 215/215 teacher tensors` (twice), `Train images: ..., val images: ...` and, with DBT, the `Dynamic Bridging Tuning: ...` line; after every epoch a `DBT: alpha_mean=...` summary is printed.

In [ ]:
env = dict(os.environ, EPOCHS=str(EPOCHS), STOP_EPOCH=str(STOP_EPOCH), INIT_EPOCH=str(INIT_EPOCH),
           MODEL_PATH=MODEL_PATH, TRAIN_SUBSET=str(TRAIN_SUBSET), VAL_SUBSET=str(VAL_SUBSET),
           BATCH_SIZE=str(BATCH_SIZE), SAVE_PERIOD=str(SAVE_PERIOD), NUM_WORKERS=str(NUM_WORKERS),
           DBT=str(DBT), DBT_MODE=DBT_MODE, DBT_USE_S=str(DBT_USE_S), DBT_USE_W=str(DBT_USE_W),
           DBT_USE_D=str(DBT_USE_D), DBT_PATH=DBT_PATH)
r = subprocess.run(['python', '-u', 'train/train_AWD.py'], env=env)
print('exit code', r.returncode)
!ls -la /kaggle/working/logs/*.pth
assert r.returncode == 0

## 5. Evaluate (only after the final epoch): mAP@0.5 on the fog and clean test sets

In [ ]:
if STOP_EPOCH >= EPOCHS:
    last = max(glob.glob('/kaggle/working/logs/ep*.pth'), key=ep_of)
    print('evaluating', last)
    for name, d in [('fog', 'datasets/voc_fog/test'), ('clean', 'datasets/voc_clean/test')]:
        e = dict(os.environ, MODEL_PATH=last, TEST_DIR=d, MAP_OUT=f'{tmp}/{name}_map_out')
        out = subprocess.run(['python', 'get_map.py'], env=e, capture_output=True, text=True).stdout
        lines = [l for l in out.splitlines() if 'AP' in l or 'mAP' in l]
        open(f'/kaggle/working/map_{name}.txt', 'w').write('\n'.join(lines))
        print('==', name); print('\n'.join(lines))
    if DBT:  # severity / IRT-trust maps for a few test images
        dbt_ck = '/kaggle/working/logs/dbt_ep%03d.pth' % ep_of(last)
        imgs = sorted(glob.glob('datasets/voc_fog/test/JPEGImages/*.jpg'))[:6]
        subprocess.run(['python', 'tools/visualize_dbt.py', '--student', last, '--dbt', dbt_ck,
                        '--out', '/kaggle/working/dbt_vis'] + imgs)
else:
    print('Not the final chunk; skipping evaluation.')